In [1]:
# Parameters
BATCH_MODE = True


# PDF Recipe Generator
## Introduction - receipe maker

**Navigation**: [Index](../README.md)

This notebook covers the main concepts and techniques of receipe maker. The educational objectives include understanding the fundamentals, hands-on practice through exercises, and analysis of results.

**Prerequisites**: basic knowledge of Python and algorithms.

This notebook orchestrates a multi-agent collaboration to:
1. Collect user constraints
2. Generate a personalized recipe
3. Produce a styled PDF

**Agents**:  
- `InputCollector` → Collects preferences  
- `RecipeGenerator` → Creates the recipe via LLM  
- `PDFGenerator` → Generates the final document

---

### Installing libraries

In [2]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')

# Cell 1 - Installation


## Global State & Configuration

In [3]:
class RecipeState:
    def __init__(self):
        self.diet: str = ""
        self.excluded_ingredients: list[str] = []
        self.guests: int = 4
        self.ingredients: list[str] = []
        self.steps: list[str] = []
        self.cooking_time: float = 0.0
        self.ready_to_generate: bool = False
        self.pdf_path: str = ""

print("Classe RecipeState définie")
print("Classe RecipeState definie")


Classe RecipeState définie
Classe RecipeState definie


### Exercise 1: Enrich the shared state with culinary preferences

The `RecipeState` class stores the diet, excluded ingredients, and number of guests. The goal is to enrich it with new fields: **difficulty level** (beginner, intermediate, expert), **maximum preparation time**, and **preferred cuisine** (French, Italian, Asian, etc.).

**Objective**: extend `RecipeState` with these new attributes and create the corresponding update methods.

**Hints**:
- # Step 1: Add the attributes `difficulty`, `max_prep_time`, and `cuisine_type` in `__init__`
- # Step 2: Create the methods `set_difficulty`, `set_max_prep_time`, and `set_cuisine`
- # Hint: follow the existing pattern of the other attributes (type hints + default value)

In [1]:
class ExtendedRecipeState(RecipeState):
    # TODO etudiant : enrichir l'etat avec les nouvelles preferences
    
    def __init__(self):
        super().__init__()
        self.difficulty: str = ""         # Etape 1 : debutant / intermediaire / expert
        self.max_prep_time: float = 0.0   # Etape 1 : temps max en minutes
        self.cuisine_type: str = ""       # Etape 1 : type de cuisine preferee
    
    def set_difficulty(self, level: str) -> str:
        result = None  # TODO etudiant : valider et assigner
        return result  # TODO etudiant : retourner confirmation
    
    def set_max_prep_time(self, minutes: float) -> str:
        result = None  # TODO etudiant : valider et assigner
        return result  # TODO etudiant : retourner confirmation
    
    def set_cuisine(self, cuisine: str) -> str:
        result = None  # TODO etudiant : assigner
        return result  # TODO etudiant : retourner confirmation

print("Exercice a completer : ExtendedRecipeState")

Exercice a completer


### Agent plugins and state modification functions

In [4]:
from semantic_kernel.functions import kernel_function
from reportlab.pdfgen import canvas

class InputCollectorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="set_diet", description="Définit le régime alimentaire")
    def set_diet(self, diet: str) -> str:
        self.state.diet = diet
        return f"Régime {diet} enregistré"

    @kernel_function(name="exclude_ingredient", description="Ajoute un ingrédient à exclure")
    def exclude_ingredient(self, ingredient: str) -> str:
        self.state.excluded_ingredients.append(ingredient)
        return f"Ingrédient {ingredient} exclu"

    @kernel_function(name="set_guests", description="Définit le nombre de convives")
    def set_guests(self, guests: int) -> str:
        self.state.guests = guests
        return f"{guests} convives prévus"

class RecipeGeneratorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state

    @kernel_function(name="submit_recipe", description="Valide la recette générée")
    def submit_recipe(self, ingredients: list[str], steps: list[str], cooking_time: float) -> str:
        self.state.ingredients = ingredients
        self.state.steps = steps
        self.state.cooking_time = cooking_time
        self.state.ready_to_generate = True
        return "Recette validée et prête pour la génération PDF"



class PDFGeneratorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="generate_pdf", description="Génère le PDF final")
    def generate_pdf(self, output_path: str) -> str:
        c = canvas.Canvas(output_path)
        c.drawString(100, 800, "Recette personnalisée")
        c.drawString(100, 780, f"Pour {self.state.guests} personnes")
        c.save()
        self.state.pdf_path = output_path
        return f"PDF généré : {output_path}"

print("Imports Semantic Kernel OK")


Imports Semantic Kernel OK


### Exercise 2: Nutrition Estimation Plugin

The current plugins handle information collection and recipe generation, but no nutritional information is provided. The goal is to add a `NutritionPlugin` that estimates the calories and macronutrients of a recipe.

**Objective**: create a `NutritionPlugin` class with an `estimate_nutrition` method annotated with `@kernel_function` that returns an estimate of calories per serving.

**Hints**:
- # Step 1: Define an approximate dictionary of calories per basic ingredient
- # Step 2: Calculate the sum of calories and divide by the number of diners
- # Hint: the dictionary can contain average values (e.g., chicken = 165 kcal/100g, rice = 130 kcal/100g)

In [1]:
class NutritionPlugin:
    """Plugin d'estimation nutritionnelle pour les recettes."""
    
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="estimate_nutrition", description="Estime les calories par portion")
    def estimate_nutrition(self, ingredients: str) -> str:
        # TODO etudiant : implementer l'estimation nutritionnelle
        # Etape 1 : dictionnaire de reference (ingredient -> kcal/100g)
        CALORIES_REF = {}  # TODO etudiant : completer le dictionnaire
        
        # Etape 2 : parser les ingredients et calculer les calories totales
        total_calories = 0  # TODO etudiant : calculer
        calories_par_portion = None  # TODO etudiant : diviser par state.guests
        
        result = None  # TODO etudiant : formater le resultat
        return result  # TODO etudiant : retourner la description nutritionnelle

print("Exercice a completer : NutritionPlugin")

Exercice a completer


### Creating agents

In [5]:
import os
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")


def build_kernel(plugin, plugin_name: str) -> Kernel:
    """Cree un kernel avec un service de chat OpenAI et le plugin fourni.

    Sans add_service, l'agent leve "No service found" des le premier appel LLM.
    """
    kernel = Kernel()
    kernel.add_service(
        OpenAIChatCompletion(
            service_id="openai",
            ai_model_id=CHAT_MODEL,
            api_key=OPENAI_API_KEY,
        )
    )
    kernel.add_plugin(plugin, plugin_name)
    return kernel


shared_state = RecipeState()

# InputCollector
input_kernel = build_kernel(InputCollectorPlugin(shared_state), "input_plugin")
input_agent = ChatCompletionAgent(
    kernel=input_kernel,
    name="InputCollector",
    instructions="""Collectez les informations utilisateur de manière structurée.
    - Demandez d'abord le régime alimentaire
    - Puis les ingrédients à exclure
    - Enfin le nombre de convives"""
)

# RecipeGenerator avec son plugin
recipe_kernel = build_kernel(RecipeGeneratorPlugin(shared_state), "recipe_plugin")
recipe_agent = ChatCompletionAgent(
    kernel=recipe_kernel,
    name="RecipeGenerator",
    instructions="""Générez des recettes en respectant les contraintes.
    - Convertir les ingrédients en liste Python
    - Structurer les étapes de préparation
    - Calculer le temps de cuisson"""
)

# PDFGenerator
pdf_kernel = build_kernel(PDFGeneratorPlugin(shared_state), "pdf_plugin")
pdf_agent = ChatCompletionAgent(
    kernel=pdf_kernel,
    name="PDFGenerator",
    instructions="""Générez un PDF professionnel:
    - Structurez en sections claires
    - Utilisez une mise en page aérée"""
)

print("Imports et configuration OK")


Imports et configuration OK


### creating the conversation with termination and/or selection criteria

In [6]:
from semantic_kernel.agents import AgentGroupChat
from pydantic import PrivateAttr

from semantic_kernel.agents.strategies import TerminationStrategy

class ReadyTerminationStrategy(TerminationStrategy):
    _state: RecipeState = PrivateAttr()

    def __init__(self, state: RecipeState, **kwargs):
        super().__init__(**kwargs)
        self._state = state

    async def should_agent_terminate(self, agent, history):
        return self._state.ready_to_generate


group_chat = AgentGroupChat(
    agents=[input_agent, recipe_agent, pdf_agent],
    termination_strategy=ReadyTerminationStrategy(shared_state),
)

print("Imports agents OK")


Imports agents OK


### Main loop

In [7]:
# Fallback: s'assurer que group_chat est défini
try:
    group_chat
except NameError:
    print("⚠️ group_chat non défini - vérifiez que la cellule précédente a été exécutée")
    group_chat = None

async def recipe_workflow():
    if group_chat is None:
        print("❌ Impossible d'exécuter: group_chat non initialisé")
        return
        
    recipe_query = (
        "Je souhaite une recette végétarienne pour 6 personnes, "
        "sans champignons ni produits laitiers."
    )
    
    group_chat.history.add_user_message(recipe_query)
    
    try:
        async for message in group_chat.invoke():
            print(f"[{message.name}] {message.content}")
            
            if shared_state.ready_to_generate:
                print("\\nTransition vers la génération PDF...")
                break
                
    except Exception as e:
        print(f"Erreur lors de la génération: {str(e)}")
    
    print("\\nProcessus terminé")

await recipe_workflow()


[InputCollector] Votre demande a bien été enregistrée. Voici un récapitulatif :

- **Régime alimentaire :** Végétarien
- **Ingrédients exclus :** Champignons, Produits laitiers
- **Nombre de convives :** 6

Je vais maintenant vous proposer une recette végétarienne adaptée à vos critères.


[RecipeGenerator] Voici une délicieuse recette végétarienne pour 6 personnes, sans champignons ni produits laitiers :

### Salade de Quinoa aux Légumes

#### Ingrédients
```python
ingredients = [
    "400g de quinoa",
    "1 courgette",
    "2 poivrons (rouge et jaune)",
    "1 boîte de pois chiches (400g)",
    "1 oignon",
    "2 gousses d'ail",
    "4 cuillères à soupe d'huile d'olive",
    "1 cuillère à café de cumin moulu",
    "1 cuillère à soupe de paprika",
    "Sel",
    "Poivre",
    "1 bouquet de coriandre fraîche"
]
```

#### Étapes de préparation
1. Rincer le quinoa sous l'eau froide, puis le cuire dans une casserole avec 800 ml d'eau bouillante pendant 15 minutes. Égoutter et laisser refroidir.
2. Dans une grande poêle, chauffer l'huile d'olive et ajouter l'oignon émincé et l'ail haché. Faire revenir jusqu'à ce qu'ils soient dorés.
3. Ajouter les poivrons coupés en dés et la courgette coupée en rondelles. Faire revenir pendant 5 à 7 minutes jusqu'à ce qu'ils soient tendres

### Exercise 3: Constraint validation before generation

The current workflow launches generation without checking that the constraints are coherent. The objective is to implement a function `valider_contraintes` that checks the coherence of user preferences before passing them on to the agents.

**Objective**: write a function that detects inconsistencies (for example: vegan diet + request for cheese, negative number of guests, excluded ingredients that are part of the diet).

**Hints**:
- # Step 1: Define the validation rules (list of conditions to check)
- # Step 2: Return a tuple `(est_valide, liste_erreurs)` to guide the user
- # Hint: use if/elif conditions for each rule and accumulate error messages

In [1]:
def valider_contraintes(state: RecipeState) -> tuple[bool, list[str]]:
    # TODO etudiant : implementer la validation des contraintes
    erreurs = []
    
    # Etape 1 : verifier la coherence du regime et des ingredients exclus
    # Etape 2 : verifier le nombre de convives, la duree, etc.
    
    est_valide = len(erreurs) == 0  # TODO etudiant : ajouter les conditions
    return est_valide, erreurs

print("Exercice a completer : validation des contraintes")

Exercice a completer


### PDF generation

In [8]:
pass

print("Workflow prêt")
print("Workflow pret")


Workflow prêt
Workflow pret


In [9]:
if shared_state.pdf_path:
    print(f"PDF généré avec succès: {shared_state.pdf_path}")
    print("Contenu de la recette:")
    print(f"- Régime: {shared_state.diet}")
    print(f"- Ingrédients: {', '.join(shared_state.ingredients)}")
else:
    print("Échec de la génération:", 
          "Aucun PDF produit malgré les tentatives")


Échec de la génération: Aucun PDF produit malgré les tentatives


## Conclusion

This notebook implemented a multi-agent collaboration to generate a personalized recipe and produce a PDF.

**Key points**:

- **Shared state**: the `RecipeState` class serves as shared memory for the three agents. Each plugin reads from and writes to this single instance, which makes it possible to pass the user constraints through to PDF generation.
- **Plugins and `@kernel_function`**: the `@kernel_function` decorator exposes Python methods as tools callable by the LLM (`set_diet`, `submit_recipe`, `generate_pdf`).
- **`AgentGroupChat` orchestration**: the `InputCollector`, `RecipeGenerator`, and `PDFGenerator` agents interact in a group conversation, each with its own kernel and chat service.
- **Termination strategy**: `ReadyTerminationStrategy` queries the shared state (`ready_to_generate`) to stop the loop as soon as the recipe is validated.
- **Service configuration**: each kernel must receive a service via `add_service` (here `OpenAIChatCompletion`), otherwise invocation fails with `No service found`.

**To go further**:

- Enhance the PDF with a styled layout (sections, images, ingredients table) via `reportlab.platypus`.
- Connect a nutritional API to calculate the recipe’s calorie intake.
- Generate an illustration of the dish with an image model and insert it into the document.